# Import


# Preset


In [ ]:
# import os
# os.environ['PATH'] += ':/Users/wenye/google-cloud-sdk/bin'

# !gcloud auth login
# !gcloud config set project final-project-1-444506

# Settings


In [1]:
from langchain_google_community.bigquery import BigQueryLoader
from google.oauth2 import service_account

CREDENTIAL_PATH = (
    "/Users/wenye/SideProjects/Data Model/Stream-Assistant/Stream-Assistant/final-project-vectorestore_credentials.json"
)
CREDENTIAL_OBJ = service_account.Credentials.from_service_account_file(filename=CREDENTIAL_PATH)

In [4]:
from datetime import datetime

# Pre Settings
PROJECT_ID = "final-project-1-444506"
REGION = "US"

DATASET = "final_project_poc"
table_name = "doc_and_vectors"

today_date = datetime.now().strftime("%Y%m%d")
TABLE = f"{table_name}_test_{today_date}"

In [5]:
from langchain_google_vertexai import VertexAIEmbeddings

embedding = VertexAIEmbeddings(model_name="textembedding-gecko@latest", project=PROJECT_ID, credentials=CREDENTIAL_OBJ)

In [6]:
TABLE

'doc_and_vectors_test_20241229'

In [7]:
DATASET

'final_project_poc'

In [8]:
from langchain_google_community import BigQueryVectorStore

store = BigQueryVectorStore(
    project_id=PROJECT_ID,
    dataset_name=DATASET,
    table_name=TABLE,
    location=REGION,
    embedding=embedding,
    credentials=CREDENTIAL_OBJ,
)

BigQuery table final-project-1-444506.final_project_poc.doc_and_vectors_test_20241229 initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=final-project-1-444506&ws=!1m5!1m4!4m3!1sfinal-project-1-444506!2sfinal_project_poc!3sdoc_and_vectors_test_20241229


In [9]:
import random
from datetime import datetime, timedelta

# Original metadata and all_tags
all_tags = [
    "technology",
    "javascript-tips",
    "world",
    "technology",
    "technology",
    "technology",
    "technology",
    "linux",
    "gpu-computing",
    "technology",
]

# Assign sources based on the rule: first 3 -> csdn, next -> medium, rest -> github
sources = ["csdn"] * 3 + ["medium"] * 4 + ["github"] * 3

# Create metadata with assigned sources
metadatas = [{"source": source} for source in sources]


# Helper function to generate random dates
def random_date(start_month, end_month, year=2024):
    start_date = datetime(year, start_month, 1)
    if end_month == 12:  # Handle December separately
        end_date = datetime(year, end_month, 31)  # Set to the last day of December
    else:
        end_date = datetime(year, end_month + 1, 1) - timedelta(days=1)
    delta = end_date - start_date
    random_days = random.randint(0, delta.days)
    return (start_date + timedelta(days=random_days)).strftime("%Y-%m-%d")


# Update metadata with random published_date
for i in range(len(metadatas)):
    if i < 5:  # First 5 are from November
        metadatas[i]["published_date"] = random_date(11, 11)
    else:  # Last 5 are from December
        metadatas[i]["published_date"] = random_date(12, 12)

# Combine tags and updated metadata for display
data = [{"tag": tag, "metadata": meta} for tag, meta in zip(all_tags, metadatas)]
import pandas as pd

df = pd.DataFrame(data)
df

,tag,metadata
0,technology,"{'source': 'csdn', 'published_date': '2024-11-..."
1,javascript-tips,"{'source': 'csdn', 'published_date': '2024-11-..."
2,world,"{'source': 'csdn', 'published_date': '2024-11-..."
3,technology,"{'source': 'medium', 'published_date': '2024-1..."
4,technology,"{'source': 'medium', 'published_date': '2024-1..."
5,technology,"{'source': 'medium', 'published_date': '2024-1..."
6,technology,"{'source': 'medium', 'published_date': '2024-1..."
7,linux,"{'source': 'github', 'published_date': '2024-1..."
8,gpu-computing,"{'source': 'github', 'published_date': '2024-1..."
9,technology,"{'source': 'github', 'published_date': '2024-1..."


In [10]:
all_tags

['technology',
 'javascript-tips',
 'world',
 'technology',
 'technology',
 'technology',
 'technology',
 'linux',
 'gpu-computing',
 'technology']

In [11]:
metadatas

[{'source': 'csdn', 'published_date': '2024-11-05'},
 {'source': 'csdn', 'published_date': '2024-11-23'},
 {'source': 'csdn', 'published_date': '2024-11-21'},
 {'source': 'medium', 'published_date': '2024-11-07'},
 {'source': 'medium', 'published_date': '2024-11-24'},
 {'source': 'medium', 'published_date': '2024-12-24'},
 {'source': 'medium', 'published_date': '2024-12-21'},
 {'source': 'github', 'published_date': '2024-12-13'},
 {'source': 'github', 'published_date': '2024-12-25'},
 {'source': 'github', 'published_date': '2024-12-16'}]

In [12]:
store.add_texts(all_tags, metadatas=metadatas)

['2c3c1f6f7362436cb72301f06f76d03d',
 '33ea85d258e642b79174ebc7fda88809',
 '9670cf2835114339ba17430753b55b8d',
 'c5820c86945740feb4790b5eaa7a4f4f',
 '3c09eb167ee74dd7bd137c1e6e378b9e',
 'b9c1d8a07beb42bfbf634ceb076b74e1',
 'd79c2855e319449ea83a317ca615186d',
 '90d32bf629664587a36adcb27dacedc7',
 'e422ff7f28ef494886bf05ceb2dcedaf',
 '067d37273ddc41cf812e453a91d62faa']

In [13]:
docs_for_tags = store.similarity_search(
    query="technology.",
    filter=(
        "published_date >= '2024-12-19' AND published_date <= '2024-12-31' "
        "AND (source = 'medium' OR source = 'github')"
    ),
    k=10,
)

docs_for_tags

[Document(metadata={'doc_id': 'b9c1d8a07beb42bfbf634ceb076b74e1', 'source': 'medium', 'published_date': '2024-12-24', 'score': 0.11537078028079432}, page_content='technology'),
 Document(metadata={'doc_id': 'd79c2855e319449ea83a317ca615186d', 'source': 'medium', 'published_date': '2024-12-21', 'score': 0.11537078028079432}, page_content='technology'),
 Document(metadata={'doc_id': 'e422ff7f28ef494886bf05ceb2dcedaf', 'source': 'github', 'published_date': '2024-12-25', 'score': 0.8908294314410925}, page_content='gpu-computing')]

In [15]:
docs_for_tags = store.similarity_search(
    query="technology.",
    filter=(
        "published_date >= '2024-12-19' AND published_date <= '2024-12-31' "
        "AND (source = 'medium' OR source = 'github')"
    ),
    k=10,
)

docs_for_tags

[Document(metadata={'doc_id': 'd79c2855e319449ea83a317ca615186d', 'source': 'medium', 'published_date': '2024-12-21', 'score': 0.11537078028079432}, page_content='technology'),
 Document(metadata={'doc_id': 'e422ff7f28ef494886bf05ceb2dcedaf', 'source': 'github', 'published_date': '2024-12-25', 'score': 0.8908294314410925}, page_content='gpu-computing')]

In [14]:
store.delete(ids=["b9c1d8a07beb42bfbf634ceb076b74e1"])

True

In [44]:
# 開始 query - 用自然語言查詢
query = "I'd like a fruit."
docs = store.similarity_search(query)
print(docs)

[Document(metadata={'doc_id': 'b93383ee027a4a218f9de38328bcc0a1', 'len': 6, 'score': 0.7318888776240622}, page_content='Banana'), Document(metadata={'doc_id': '403919b8924345b39c81fa6456526229', 'len': 9, 'score': 0.7515836427353295}, page_content='Pineapple'), Document(metadata={'doc_id': '221d8d6ce0674fc484c937e1bd04418a', 'len': 18, 'score': 0.7886588511500245}, page_content='Apples and oranges'), Document(metadata={'doc_id': 'd2df007a89d341f387cf0715a032ed98', 'len': 18, 'score': 0.8782809421845853}, page_content='Cars and airplanes'), Document(metadata={'doc_id': '23575fd46c0345d6a5ef2897c4ce6f92', 'len': 5, 'score': 0.8841776323526317}, page_content='Train')]


In [ ]:
# 開始 query - 用向量查詢
query_vector = embedding.embed_query(query)
docs = store.similarity_search_by_vector(query_vector, k=2)
print(docs)

[Document(metadata={'doc_id': '32135af40829490989f8130bc78c6010', 'len': 6, 'score': 0.7318888776240622}, page_content='Banana'), Document(metadata={'doc_id': '6df7e0b722fc440fa68bf2b255f36831', 'len': 6, 'score': 0.7318888776240622}, page_content='Banana')]


In [ ]:
# Dictionary-based Filters
# This should only return "Banana" document.

# 加入 filters
docs = store.similarity_search_by_vector(query_vector, filter={"len": 6})
print(docs)

[Document(metadata={'doc_id': '32135af40829490989f8130bc78c6010', 'len': 6, 'score': 0.7318888776240622}, page_content='Banana'), Document(metadata={'doc_id': '6df7e0b722fc440fa68bf2b255f36831', 'len': 6, 'score': 0.7318888776240622}, page_content='Banana')]


In [41]:
# 可以一次放多個
results = store.batch_search(
    embeddings=None,  # can pass embeddings or
    queries=["我要問機票哪裡買", "何時出新手機"],  # can pass queries
)

results

[[[Document(metadata={'doc_id': '5e30abc205e648f388c377e980f49afb', 'len': 1, 'score': 0.7874963242790278}, page_content='some text'),
   0.7874963242790278],
  [Document(metadata={'doc_id': 'bb2befe261e4411c92ef90de44fcb173', 'len': 5, 'score': 0.7945497565425332}, page_content='Train'),
   0.7945497565425332],
  [Document(metadata={'doc_id': 'c491c0964df140bba16bc910348145f6', 'len': 5, 'score': 0.7945497565425332}, page_content='Train'),
   0.7945497565425332],
  [Document(metadata={'doc_id': '034afe5be09840f3aa1b095b2e93ce55', 'len': 9, 'score': 0.8095500375860955}, page_content='Pineapple'),
   0.8095500375860955],
  [Document(metadata={'doc_id': 'd233251183be4624860e806c1c3e4abf', 'len': 9, 'score': 0.8095500375860955}, page_content='Pineapple'),
   0.8095500375860955]],
 [[Document(metadata={'doc_id': '5e30abc205e648f388c377e980f49afb', 'len': 1, 'score': 0.7874963242790278}, page_content='some text'),
   0.7874963242790278],
  [Document(metadata={'doc_id': 'bb2befe261e4411c92ef

In [ ]:
# 加入 embedding - 要傳入 query, embeddings, metadatas
items = ["some text"]
embs = embedding.embed(items)

ids = store.add_texts_with_embeddings(texts=["some text"], embs=embs, metadatas=[{"len": 1}])